# <b><span style='color: #196f3d '>| PlantTraits2024:</span>  Hybrid Model [Train]</b> 
The task is to predict the mean value of 6 plant features based on plant images and tabular data. Training data contains both the mean value and std of plant traits. We will refer to predicting the mean as the main task and predicting the std as the auxillary task. 

This notebook is based on [this notebook](https://www.kaggle.com/code/hdjojo/modified-planttraits2024-eda-training) and [this notebook](https://www.kaggle.com/code/mohammedessam97/planttraits2024-pytorch-starter-notebook). The layout from [Moths WaveNet Starter](https://www.kaggle.com/code/alejopaullier/hms-wavenet-pytorch-train). This notebook is used for training, the inference notebook can be found [here](https://www.kaggle.com/code/jasperpieterse/planttraits2024-inference).

### <b><span style='color: #196f3d '>Table of Contents</span></b> <a class='anchor' id='top'></a>
<div style=" background-color: #ecf0f1 ; padding: 13px 13px; border-radius: 8px; color: white">
<li><a href="#import_libraries">Import Libraries</a></li>
<li><a href="#Configuration">Configuration</a></li>
<li><a href="#load_data">Load Data</a></li>
<li><a href="#outlier_removal">Outlier Removal</a></li>
<li><a href="#crossvalidation">Cross Validation</a></li>    
<li><a href="#augmentations">Augmentations</a></li>
<li><a href="#dataset">Dataset</a></li>
<li><a href="#dataloader">DataLoader</a></li>
<li><a href="#model">Model</a></li>
<li><a href="#optimizer">Optimizer & Scheduler</a></li>
<li><a href="#loss">Loss Function</a></li>
<li><a href="#train">Train Model</a></li>
<li><a href="#train">Score</a></li>
    
</div>

# <b><span style='color: #196f3d '>|</span> Import Libraries</b><a class='anchor' id='import_libraries'></a> [↑](#top) 

***

Import and install all the required libraries for this notebook.

In [ ]:
import os
import cv2
import torch
import glob
import time
import joblib
import torchmetrics
import timm
import warnings
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl
import albumentations as A
import imageio.v3 as imageio
import psutil
import math

from glob import glob
from PIL import Image
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from albumentations.pytorch import ToTensorV2
from torchmetrics.regression import R2Score
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import OneCycleLR
from torchvision.models import efficientnet
from sklearn.model_selection import StratifiedKFold

tqdm.pandas()

# <b><span style='color: #196f3d '>|</span> Configuration</b><a class='anchor' id='Configuration'></a> [↑](#top) 

***

In [ ]:
class CFG:
    IMAGE_SIZE = 384
#     BACKBONE = 'efficientnet_b0'      # Name of pretrained EfficientNet model (B0 - B7)
    BACKBONE = 'swin_large_patch4_window12_384.ms_in22k_ft_in1k'
    N_EPOCHS = 6 
    BATCH_SIZE = 10 
    LR_MAX = 1e-4
    WEIGHT_DECAY = 0.01
    DROPOUT = 0.2
    NUM_CLASSES = 6 
    NUM_FOLDS = 5 
    FOLD = 0 # Which fold to set as validation set
    TARGET_COLUMNS = ['X4_mean', 'X11_mean', 'X18_mean', 'X50_mean', 'X26_mean', 'X3112_mean'] #don't change order 
    AUX_TARGET_COLUMNS = ['X4_sd', 'X11_sd', 'X18_sd', 'X50_sd', 'X26_sd', 'X3112_sd']
    SEED = 42  
    CREATE_DATAFRAME = False #Whether to create the data or load it
    USE_SUBSET_DATA = False #Use 1/3th of the data for faster experimentation
    CNN_FEATURES = 64 #Amount of features the CNN should output to the heads
    TABULAR_FEATURES = 32 #Amount of features the Dense layer should output to the heads

# Set a seed using Pytorch Lightning
pl.seed_everything(CFG.SEED, workers=True)
# warnings.filterwarnings('ignore')

# Set base path
BASE_PATH = "/kaggle/input/planttraits2024"
DATAFRAME_PATH = "/kaggle/input/planttraits2024-dataframes"
REDUCED_DATAFRAME_PATH = '/kaggle/input/planttraits2024-reduced-dataset'
PRETRAINED_REGRESSOR_PATH = 'path/to/pretrained_regressor.pth'
PRETRAINED_IMAGE_PATH = 'path/to/pretrained_image.pth'

# <b><span style='color: #196f3d '>|</span> Helper Functions</b><a class='anchor' id='load_data'></a> [↑](#top) 

In [ ]:
def scale_with_exclusion(data, scaler, IsTrain = True):
    """Applies a standard scaler, maintaining the -1 values of the data to be masked later"""
    scaled_data = np.zeros(data.shape, dtype=np.float32)
    mask = (data != -1)  #Mask non -1 values
    
    # Apply the scaler only to non -1 values
    if IsTrain:
        scaled_data[mask] = scaler.fit_transform(data[mask].reshape(-1, 1)).flatten()
    else:
        scaled_data[mask] = scaler.transform(data[mask].reshape(-1, 1)).flatten()
    
    scaled_data[~mask] = -1 # Restore -1 values
    
    return scaled_data

# <b><span style='color: #196f3d '>|</span> Load Data</b><a class='anchor' id='load_data'></a> [↑](#top) 

We either load the data or create dataframe from metadata and embedd raw image bytes in it. We also added to option to load a subset of the data, based on a stratified split.

In [ ]:
df = pd.read_csv(f'{BASE_PATH}/train.csv')

if CFG.USE_SUBSET_DATA:
    #SPLIT DATA IN 3 STRATIFIED FOLDS 
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) #split data in 10 Stratified Folds
    # Create separate bin for each trait
    for i, trait in enumerate(CFG.TARGET_COLUMNS):
        bin_edges = np.percentile(df[trait], np.linspace(0, 100, CFG.NUM_FOLDS + 1))
        df[f"bin_{i}"] = np.digitize(df[trait], bin_edges)
    df["final_bin"] = df[[f"bin_{i}" for i in range(len(CFG.TARGET_COLUMNS))]].astype(str).agg("".join, axis=1)

    # Perform the stratified split using final bin
    df = df.reset_index(drop=True)
    for fold, (train_idx, valid_idx) in enumerate(skf.split(df, df["final_bin"])):
        df.loc[valid_idx, "fold"] = fold

    #SELECT ONE FOLD
    df = df[df.fold == CFG.FOLD]
    
    # Drop the bin columns from the dataframe
    bin_columns = [f"bin_{i}" for i in range(len(CFG.TARGET_COLUMNS))]
    df = df.drop(columns= bin_columns +['final_bin','fold'])
    
if CFG.CREATE_DATAFRAME:
    df['image_path'] = df['id'].apply(lambda s: f'{BASE_PATH}/train_images/{s}.jpeg')     # add image path info to df
    df['image_bytes'] = df['image_path'].progress_apply(lambda fp: open(fp, 'rb').read()) # embedd image information in dataframe
    df.loc[:, CFG.AUX_TARGET_COLUMNS] = df.loc[:, CFG.AUX_TARGET_COLUMNS].fillna(-1)      # replace std NaN values with -1 (stds are always >0)
    df.to_pickle('df.pkl')

    test_df = pd.read_csv(f'{BASE_PATH}/test.csv')
    test_df['image_path'] = test_df['id'].apply(lambda s: f'{BASE_PATH}/test_images/{s}.jpeg') 
    test_df['image_bytes'] = test_df['image_path'].progress_apply(lambda fp: open(fp, 'rb').read()) 
    test_df.to_pickle('test_df.pkl')

#load from file, depending on whether we use a subset or full data
else:
    if CFG.USE_SUBSET_DATA:
        df = pd.read_pickle(f'{REDUCED_DATAFRAME_PATH}/df.pkl')
        test_df = pd.read_pickle(f'{REDUCED_DATAFRAME_PATH}/test_df.pkl')
    else:
        df = pd.read_pickle(f'{DATAFRAME_PATH}/df.pkl')
        test_df = pd.read_pickle(f'{DATAFRAME_PATH}/test_df.pkl')
        
#STORE TARGET COLLUMNS
CFG.FEATURE_COLS = test_df.columns[1:-2].tolist() #leave out img-path and image data

# <b><span style='color: #196f3d '>|</span> Outlier & Nonunique Sample Removal</b><a class='anchor' id='outlier_removal'></a> [↑](#top) 

Our training data contains quite some bad outliers. According to the competition host, [these outliers have been removed from the test set](https://www.kaggle.com/competitions/planttraits2024/discussion/483414), however, they remain still in our training and validation dataframe. So before splitting into training and validation, we should remove these outliers.


In [ ]:
# REMOVE OUTLIERS FROM MAIN AND TABULAR FEATURES FROM TRAINING DATA
for column in (CFG.TARGET_COLUMNS + CFG.AUX_TARGET_COLUMNS):
    upper_quantile = df[column].quantile(0.985)
    df = df[(df[column] < upper_quantile)]
    df = df[(df[column] > 0) | (df[column] == -1)] # remove negative values except -1

# <b><span style='color: #196f3d '>|</span> Validation</b><a class='anchor' id='crossvalidation'></a> [↑](#top) 

***

We will split the training data into `5` stratified fold and take one as our validation set and the other four as training set.


In [ ]:
skf = StratifiedKFold(n_splits=CFG.NUM_FOLDS, shuffle=True, random_state=42)

# Create separate bin for each trait
for i, trait in enumerate(CFG.TARGET_COLUMNS):
    bin_edges = np.percentile(df[trait], np.linspace(0, 100, CFG.NUM_FOLDS + 1))
    df[f"bin_{i}"] = np.digitize(df[trait], bin_edges)
df["final_bin"] = df[[f"bin_{i}" for i in range(len(CFG.TARGET_COLUMNS))]].astype(str).agg("".join, axis=1)
df["fold"] = -1  # Initialize fold column

# Perform the stratified split using final bin
df = df.reset_index(drop=True)
for fold, (train_idx, valid_idx) in enumerate(skf.split(df, df["final_bin"])):
    df.loc[valid_idx, "fold"] = fold
    
# Drop the bin columns from the dataframe
bin_columns = [f"bin_{i}" for i in range(len(CFG.TARGET_COLUMNS))]
df = df.drop(columns= bin_columns +['final_bin'])
    
# Create a train and a validation dataframe based on splits
sample_df = df.copy()
train_df = sample_df[sample_df.fold != CFG.FOLD].reset_index(drop=True)
valid_df = sample_df[sample_df.fold == CFG.FOLD].reset_index(drop=True)

train_df.head()

# <b><span style='color: #196f3d '>|</span> Preprocessing</b><a class='anchor' id='preprocessing'></a> [↑](#top) 
We log-scale the tabular **features**. We then normalize both the **features** and the **targets**. We will use the following notation:
- **(Main) Features:** The images
- **Tabular Features:** The Tabular data 
- **(Main) Targets:** The means of the 6 plant trait features.
- **Auxillary Targets:** The stds of the 6 plant trait features. 

Note 1: We cannot log-scale the targets because the R2 metric punishes absolute differences instead of relative differences. Log-scaling the targets thus gives a false R2 score on validation.

Note 2: Although it's essential to apply the same preprocessing steps (like scaling) to both training and validation data to maintain consistency, we fit the scaler on the training data only, to ensure that the statistics (mean and standard deviation) used for scaling are derived exclusively from the data the model sees during the training process.


In [ ]:
#STORE MAIN FEATURES
train_features = train_df['image_bytes'].values
valid_features = valid_df['image_bytes'].values
test_features  = test_df['image_bytes'].values

# LOG-SCALE TABULAR FEATURES
train_tab_features = np.log1p(train_df[CFG.FEATURE_COLS].values)
valid_tab_features = np.log1p(valid_df[CFG.FEATURE_COLS].values)
test_tab_features = np.log1p(test_df[CFG.FEATURE_COLS].values)

#NORMALIZATION OF MAIN FEATURES IS HANDLED BY DATALOADER

#NORMALIZE TABULAR FEATURES
TAB_FEATURE_SCALER = StandardScaler()
train_tab_features = TAB_FEATURE_SCALER.fit_transform(train_tab_features)
valid_tab_features = TAB_FEATURE_SCALER.transform(valid_tab_features)
test_tab_features = TAB_FEATURE_SCALER.transform(test_tab_features) 

# NORMALIZE MAIN TARGETS
TARGET_SCALER = StandardScaler()
train_targets = TARGET_SCALER.fit_transform(train_df[CFG.TARGET_COLUMNS].values)
valid_targets = TARGET_SCALER.transform(valid_df[CFG.TARGET_COLUMNS].values)

# NORMALIZE AUXILLARY TARGETS
AUX_TARGET_SCALER = StandardScaler()
train_aux_targets = scale_with_exclusion(train_df[CFG.AUX_TARGET_COLUMNS].values, AUX_TARGET_SCALER, IsTrain = True)
valid_aux_targets = scale_with_exclusion(valid_df[CFG.AUX_TARGET_COLUMNS].values, AUX_TARGET_SCALER, IsTrain = False)

print('N_TRAIN_SAMPLES:', len(train_df),'N_VALID_SAMPLES:', len(valid_df), 'N_TEST_SAMPLES:', len(test_df))

In [ ]:
# Check whether the converted NaN values retained their -1 values
num_negative_ones_train = np.count_nonzero(train_aux_targets == -1)
print(f"Number of -1 values in train_aux_targets: {num_negative_ones_train}")

num_negative_ones_valid = np.count_nonzero(valid_aux_targets == -1)
print(f"Number of -1 values in valid_aux_targets: {num_negative_ones_valid}")

# <b><span style='color: #196f3d '>|</span>Augmentations</b><a class='anchor' id='augmentations'></a> [↑](#top) 

***
Functions of possible augmentations to apply to our data. Will be used in creating the dataset. We also preprocess the images by adding a normalization to match the stastics of ImageNet and scale pixel values to the [0,1] range.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406] #statistics of image_net photos, used to normalize images
IMAGENET_STD = [0.229, 0.224, 0.225] 

TRAIN_TRANSFORMS = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomSizedCrop(
            [int(0.85*CFG.IMAGE_SIZE), CFG.IMAGE_SIZE],
            CFG.IMAGE_SIZE, CFG.IMAGE_SIZE, w2h_ratio=1.0, p=0.75),
        A.Resize(CFG.IMAGE_SIZE, CFG.IMAGE_SIZE),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.25),
        A.ImageCompression(quality_lower=85, quality_upper=100, p=0.25),
        A.ToFloat(), #float32
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=1), #normalization to match the statistics used to pre-train the EfficientNet model
        ToTensorV2(),
    ])

TEST_TRANSFORMS = A.Compose([
        A.Resize(CFG.IMAGE_SIZE, CFG.IMAGE_SIZE),
        A.ToFloat(),#float32
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=1), #normalization to match the statistics used to pre-train the EfficientNet model
        ToTensorV2(),
    ])

# <b><span style='color: #196f3d '>|</span> Dataset</b><a class='anchor' id='dataset'></a> [↑](#top) 

***

Create a custom `Dataset` to load data. Here we can easily implement custom loading mechanisms, data pre-processing, and augmentation techniques. Also allows us to use the Pytorch Dataloader, streamlining data loading and batching.

In [ ]:
class PlantDataset(Dataset):
    def __init__(self, image_bytes, tab_features, targets = None, aux_targets = None, transforms=None):
        self.image_bytes = image_bytes
        self.tab_features = tab_features
        self.targets = targets
        self.aux_targets = aux_targets
        self.transforms = transforms

    def __len__(self):
        return len(self.image_bytes)

    def __getitem__(self, idx):
        #read the image data directly from its JPEG-encoded byte representation
        image = self.transforms(image=imageio.imread(self.image_bytes[idx]))['image'] #Access the the modified image array with shape (size,size,3) within the Albumentations dictionary 
        tab_feature = self.tab_features[idx]
        
        target = torch.tensor(self.targets[idx])
        aux_target = torch.tensor(self.aux_targets[idx], dtype=torch.float32)
        
        return {'images': image, 'tab_features': tab_feature}, (target, aux_target)

# <b><span style='color: #196f3d '>|</span> DataLoader</b><a class='anchor' id='dataloader'></a> [↑](#top) 

***


In [ ]:
train_dataset = PlantDataset(train_features, train_tab_features, train_targets, train_aux_targets, TRAIN_TRANSFORMS)

train_dataloader = DataLoader(
        train_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=psutil.cpu_count(),
)

valid_dataset = PlantDataset(valid_features, valid_tab_features, valid_targets, valid_aux_targets, TEST_TRANSFORMS)

valid_dataloader = DataLoader(
        valid_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=psutil.cpu_count(),
)

#for the test data, the 'targets' are just the image ids
test_dataset = PlantDataset(test_features, test_tab_features, test_df['id'].values, test_df['id'].values, TEST_TRANSFORMS)

test_dataloader = DataLoader(
        test_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=psutil.cpu_count(),
)

## Dataset Check

Let's visualize some samples and their associated targets from the dataset.

In [ ]:
# Get a batch of data
inps, tars = next(iter(train_dataloader))
imgs = inps["images"]
num_imgs, num_cols = 8, 4

# Convert PyTorch tensors to NumPy arrays
imgs_np = imgs.numpy()
tars_np = tars[0].numpy()

plt.figure(figsize=(4 * num_cols, num_imgs // num_cols * 5))

for i, (img, tar) in enumerate(zip(imgs_np[:num_imgs], tars_np[:num_imgs])):
    plt.subplot(num_imgs // num_cols, num_cols, i + 1)

    # Normalize the image to [0, 1]
    img = (img - img.min()) / (img.max() - img.min() + 1e-4)

    formatted_tar = "\n".join(
        [
            ", ".join(
                f"{name.replace('_mean','')}: {val:.2f}"
                for name, val in zip(CFG.TARGET_COLUMNS[j : j + 3], tar[j : j + 3])
            )
            for j in range(0, len(CFG.TARGET_COLUMNS), 3)
        ]
    )

    plt.imshow(img.transpose(1, 2, 0))  # Transpose to (height, width, channels)
    plt.title(f"[{formatted_tar}]")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Check Dataloader

One of the most overlooked speed components is preprocess and dataloader. Try running your dataloader without training a model. For example, time how long does the following take:

In [ ]:
# %%time
# for batch in train_dataloader:
#     pass

This should be lightning fast. If it is not, find the bottlenecks (disk reading, repeated operations, inefficient operations, etc) and remove them. Also utilize CPU/GPU multiprocessing in dataloader. When we train a model on GPU we want to feed large batch sizes the fastest we can because GPUs want lots of data quickly. If we run nvidia-smi we should see 100% GPU utilization. Anything less indicates a dataloader bottleneck and unnecessary slow down.

# <b><span style='color: #196f3d '>|</span> Model</b><a class='anchor' id='model'></a> [↑](#top) 

***
We separately extract features from the image data and the tabular data and concatenate these. These are then passed through two different heads. The main head predicts the mean values based on these features, while the auxillary head predicts the stds. The main head has no activiations, while the auxillary head has `relu` activations because we are estimating the **standard deviation** of plant traits, which is always **positive**. For the final prediction, the results of both heads get combined. We assign more weight to the `head` than the `aux_head` since it is our main task, and our evaluation metric is calculated for the mean values, not the stds.

In [ ]:
class Regressor(nn.Module):
    def __init__(self, input_size):
        super(Regressor, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, CFG.TABULAR_FEATURES)  
        self.relu_out = nn.ReLU()
        
        # output layers 
        self.head = nn.Linear(CFG.TABULAR_FEATURES, CFG.NUM_CLASSES)
        self.aux_head = nn.Linear(CFG.TABULAR_FEATURES, CFG.NUM_CLASSES)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu_out(x)
        
        # Output layer
        out1 = self.head(x)
        out2 = self.aux_head(x)

        return {'head': out1, 'aux_head': out2}

In [ ]:
class CustomModel(nn.Module):
    def __init__(self):
        super(CustomModel, self).__init__()
        
        # Branch for the images
        self.backbone = timm.create_model(CFG.BACKBONE, num_classes=CFG.CNN_FEATURES, pretrained=True)
        self.dropout_img = nn.Dropout(CFG.DROPOUT)

        # Branch for tabular feature input using the pre-trained Regressor
        self.classifier = Regressor(input_size=len(CFG.FEATURE_COLS), output_size=CFG.TABULAR_FEATURES)
        
        # Output layers 
        self.head = nn.Linear(CFG.CNN_FEATURES + CFG.TABULAR_FEATURES, CFG.NUM_CLASSES)
        self.aux_head = nn.Linear(CFG.CNN_FEATURES + CFG.TABULAR_FEATURES, CFG.NUM_CLASSES)

    def forward(self, img, feat):
        # Image branch
        x1 = self.backbone(img)
        x1 = self.dropout_img(x1.flatten(1))

        # Feature branch using the pre-trained Regressor
        x2 = self.classifier(feat)

        # Concatenate both branches
        concat = torch.cat([x1, x2], dim=1)
        
        # Output layer
        out1 = self.head(concat)
        out2 = F.relu(self.aux_head(concat))

        return {'head': out1, 'aux_head': out2}

# <b><span style='color: #196f3d '>|</span> Custom Loss Functions</b><a class='anchor' id='loss'></a> [↑](#top) 

***

The evaluation metric in this competition is $R^2$. We will probably use a different Loss function though. Some samples don't have target targets, we will exclude them from the loss calculation using `use_mask` argument.

In [ ]:
class R2Loss(nn.Module):
    """Custom R2 Metric that can mask the NaN values. Equivalent to the Torch Implementation."""
    def __init__(self, use_mask=False):
        super(R2Metric, self).__init__()
        self.use_mask = use_mask

    def forward(self, y_pred, y_true):
        if self.use_mask:
            mask = (y_true != -1)
            y_true = torch.where(mask, y_true, torch.zeros_like(y_true))
            y_pred = torch.where(mask, y_pred, torch.zeros_like(y_pred))

        SS_res = torch.sum((y_true - y_pred)**2, dim=0)  # (B, C) -> (C,)
        SS_tot = torch.sum((y_true - torch.mean(y_true, dim=0))**2, dim=0)  # (B, C) -> (C,)
        r2_loss = SS_res / (SS_tot + 1e-6)  # (C,)
        return torch.mean(r2_loss) #Return -R2 to minimize


class SmoothL1Loss(nn.Module):
    def __init__(self, reduction='mean', use_mask = False):
        super(SmoothL1Loss, self).__init__()
        self.reduction = reduction
        self.use_mask = use_mask
        self.smooth_l1_loss = nn.SmoothL1Loss(reduction='none')  # Use 'none' to handle reduction manually

    def forward(self, inputs, targets):
        """
        Compute the SmoothL1Loss with an optional mask.
        """
        losses = self.smooth_l1_loss(inputs, targets)
        
        if self.use_mask:
            mask = (targets != -1)
            losses = losses * mask  # Apply mask by element-wise multiplication

        if self.reduction == 'mean':
            return losses.sum() / mask.sum() if self.use_mask else losses.mean() # Only consider masked elements for the mean
        elif self.reduction == 'sum':
            return losses.sum()
        else:
            return losses

# <b><span style='color: #196f3d '>|</span> LR Scheduler</b><a class='anchor' id='optimizer'></a> [↑](#top) 

***

A well-structured learning rate schedule is essential for efficient model training, ensuring optimal convergence and avoiding issues such as overshooting or stagnation.

In [ ]:
#Get stats to initalize scheduler
CFG.N_TRAIN_SAMPLES = len(train_df)
CFG.N_STEPS_PER_EPOCH = (CFG.N_TRAIN_SAMPLES // CFG.BATCH_SIZE)
CFG.N_STEPS = CFG.N_STEPS_PER_EPOCH * CFG.N_EPOCHS + 1

def get_lr_scheduler(optimizer):
    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer=optimizer,
        max_lr=CFG.LR_MAX,
        total_steps=CFG.N_STEPS,
        pct_start=0.1,
        anneal_strategy='cos',
        div_factor=1e1,
        final_div_factor=1e1,
    )

# <b><span style='color: #196f3d '>|</span> Train Model</b><a class='anchor' id='train'></a> [↑](#top) 

***

In [ ]:
# Initialize the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CustomModel()
model.to(device)

# Load the pre-trained Image Model weights
image_state_dict = torch.load(PRETRAINED_IMAGE_PATH)
model.backbone.load_state_dict(image_state_dict)

# Load the pre-trained Tabular Regressor weights
regressor_state_dict = torch.load(PRETRAINED_REGRESSOR_PATH)
model.classifier.load_state_dict(regressor_state_dict)

# Set optimizer
optimizer = torch.optim.AdamW(
    params=model.parameters(),
    lr=CFG.LR_MAX,
    weight_decay=CFG.WEIGHT_DECAY,
)

# Initialize the loss functions
#R2 Losses
LOSS_MAIN = R2Loss(use_mask = False)
LOSS_AUX  = R2Loss(use_mask = True)

#L1 Losses
# LOSS_MAIN = nn.SmoothL1Loss()
# LOSS_AUX  = SmoothL1Loss(use_mask = True)


#Initialize the Metric
R2_FN = R2Score(num_outputs=CFG.NUM_CLASSES, multioutput='uniform_average').to('cuda')

# Set lr_scheduler
lr_scheduler = get_lr_scheduler(optimizer)

# Loss weights
weight_head = 1.0
weight_aux_head = 0.0

# Model checkpoint
best_model_path = "best_model.pth"
final_model_path = "final_model.pth"
best_r2_score = -float('inf')

#=====================TRAINING=====================
for epoch in range(CFG.N_EPOCHS):
    # Training
    total_train_r2 = 0.0
    total_loss     = 0.0
    train_batches  = 0

    model.train()
    for batch_idx, (inputs_dict, (targets, aux_targets)) in enumerate(tqdm(train_dataloader)):
        # Move data to GPU
        inputs_images = inputs_dict['images'].to(device, dtype=torch.float32)
        inputs_features = inputs_dict['tab_features'].to(device, dtype=torch.float32)
        
        targets = targets.to(device, dtype=torch.float32)
        aux_targets = aux_targets.to(device, dtype=torch.float32)
        t_start = time.perf_counter_ns()

        # Forward pass
        preds = model(inputs_images, inputs_features)

        # Compute losses
        weighted_loss_head = weight_head * LOSS_MAIN(preds['head'], targets)
        weighted_loss_aux_head = weight_aux_head * LOSS_AUX(preds['aux_head'], aux_targets)
        loss = weighted_loss_head + weighted_loss_aux_head

        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Reset gradient and update LR scheduler
        optimizer.zero_grad()
        lr_scheduler.step()

        # Update the R2 metric
        total_train_r2 += R2_FN(preds['head'], targets).item() 
        total_loss += loss
        train_batches += 1

    # Compute the average training R2 score
    avg_train_r2 = total_train_r2 / train_batches
    avg_loss = total_loss / train_batches

#=====================VALIDATION=====================
    model.eval()
    total_val_r2 = 0.0
    val_batches = 0

    with torch.no_grad():
        for val_batch_idx, (val_inputs_dict, (val_targets, val_aux_targets)) in enumerate(tqdm(valid_dataloader)):
            val_inputs_images = val_inputs_dict['images'].to(device, dtype=torch.float32)
            val_inputs_features = val_inputs_dict['tab_features'].to(device, dtype=torch.float32)
          
            val_targets = val_targets.to(device, dtype=torch.float32)
            val_aux_targets = val_aux_targets.to(device, dtype=torch.float32)

            val_preds = model(val_inputs_images, val_inputs_features)

            # Compute the R2 metric for validation
            total_val_r2 += R2_FN(val_preds['head'], val_targets).item()
            val_batches += 1

    # Compute the average validation R2 score
    avg_val_r2 = total_val_r2 / val_batches
    
    #Print Information
    print(  f'EPOCH {epoch+1}/{CFG.N_EPOCHS}, {CFG.N_STEPS_PER_EPOCH * epoch +1:04d}/{CFG.N_STEPS} | ' + 
            f'TRAIN LOSS: {avg_loss:.4f}, TRAIN R2: {avg_train_r2:.4f}, VAL R2: {avg_val_r2:.4f} | ' +
            f'EPOCH TIME : {(time.perf_counter_ns()-t_start)*1e-9:.3f}s, LR: {lr_scheduler.get_last_lr()[0]:.2e} \n',
         )

    # Save the best model based on validation R2 score
    if avg_val_r2 > best_r2_score:
        best_r2_score = avg_val_r2
        torch.save(model.state_dict(), best_model_path)

#Save the final model
torch.save(model.state_dict(), final_model_path)
print(f'=========FINISHED TRAINING========\nBEST R2 SCORE: {best_r2_score}')